# AML_06 — Livrer un système et défendre une expérience

*Conception pédagogique : Mathieu Weill. Usage personnel d’apprentissage ; droits et crédits tiers conservés.*

**Compétition de travail : [fraud](https://www.kaggle.com/competitions/playground-series-s6e11)**

**Question : Un autre étudiant peut-il reproduire notre résultat ?**

7 heures pédagogiques, hors pauses. Les exemples sont des jalons techniques courts ; l’essentiel du temps est consacré aux exercices, revues et expériences. **Version étudiant.**

**[Repère de compréhension : Livrer et surveiller un système](../../../course/chapitres/AML_06/README.md)**


## Lecture accompagnée

Avant le code, lire avec le professeur le [chapitre 6 du cours rédigé](../../../course/chapitres/AML_06/cours.pdf), section par section. Répondre à une question, corriger ensemble, puis retrouver le même raisonnement dans le notebook. Le PDF complet est disponible dans le dossier `student/lectures`.

## Parcours guidé — Livrer une prédiction que quelqu’un peut utiliser

**Objectif minimal :** Un membre reproduit le run et un autre explique les contrôles. Produire le CSV aligné, la fiche modèle et une limite reconnue.

**Idée à retenir :** Un modèle livré est un contrat de données, de prétraitement, de sortie et de reproduction.

Nous avançons en cinq étapes : **je montre → nous faisons ensemble → vous reproduisez → vous changez une chose → vous expliquez**. Ne pas lancer les autres cellules avant le signal de l’enseignant. Le premier exemple est volontairement minuscule et distinct des données de compétition.

### Exemple au tableau

Écrire le sample submission dans l’ordre des IDs : 18, 12, 27.

Un autre tableau donne les prédictions dans l’ordre 12, 18, 27 : 0,9 ; 0,1 ; 0,3. Affecter la colonne sans jointure produit une erreur silencieuse.

Résultat correctement aligné : 18→0,1 ; 12→0,9 ; 27→0,3.

Écrire la chaîne : entrée → contrôle du schéma → préparation → modèle → classes/probabilités → fichier aligné → contrôle. Le test de l’école n’est pas le leaderboard, et la livraison locale n’est pas une soumission Kaggle.

### Répondez avant de lancer Python

1. Pourquoi l’ordre des IDs peut-il invalider une soumission ?
2. Que manque-t-il à un fichier de poids seul ?
3. Que faut-il figer avant l’évaluation finale ?
4. Que fait-on après la clôture de la compétition ?

## Préparation et limites

Lire README.md, compétitions et règles. Les données officielles sont téléchargées personnellement et placées dans data/competitions/<slug>/. Aucun remplacement silencieux par des données simulées. Poids préentraînés à préparer avant NLP_02/CV_01/CV_02. Les limites de calcul et d’échanges dépendent du concours et du compte.

## Notions du jour

Avant une soumission, fixer modèle, prétraitement et choix avec la validation. Réentraîner ensuite sur les données de développement retenues. Le laboratoire peut limiter le nombre de lignes pour un essai CPU : cela doit être déclaré, pas présenté comme l’entraînement intégral.

La compétition de novembre clôture le **30 novembre 2026 à 23:59 UTC**. Si J6 a lieu le 17 décembre, cette partie est une reproduction et un bilan après clôture. Ne pas annoncer une soumission classée après la date. Une nouvelle compétition de décembre ne sera utilisée qu’après lecture de ses règles et adaptation réelle du contrat.

MLOps : schéma d’entrée, classes, dépendances, versions, artefacts, suivi et retour arrière. Le déploiement ne transforme pas le leaderboard en test indépendant.

In [ ]:
from pathlib import Path
import sys,os,json,time
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
HERE=Path.cwd()
ROOT=next((p if (p/'mllab/competition.py').exists() else p/'student') for p in [HERE,*HERE.parents] if (p/'mllab/competition.py').exists() or (p/'student/mllab/competition.py').exists())
sys.path.insert(0,str(ROOT))
from mllab.competition import SPECS,read_competition,protocol,run_competition,binary_scores,get_folder
KEY='fraud';SPEC=SPECS[KEY]
QUICK=os.environ.get('MLLAB_QUICK','0')=='1'
LIMIT=int(os.environ.get('MLLAB_LIMIT','1000' if QUICK else '15000'))
IMAGE_LIMIT=int(os.environ.get('MLLAB_IMAGE_LIMIT','24' if QUICK else '240'))
plt.rcParams.update({'figure.figsize':(10,4),'axes.spines.top':False,'axes.spines.right':False})
print('Compétition :',SPEC['slug'],'métrique :',SPEC['metric'],'sens :',SPEC['direction'])
if KEY!='catsdogs':a,test,sample,protocol_record=protocol(KEY)
else:
 from mllab.vision_competition import image_protocol
 image_protocol(limit=IMAGE_LIMIT)
print('Données officielles trouvées. Profil rapide :',QUICK,'; aucun score Kaggle inventé.')

### Retrouver le tableau en Python

Lire les entrées ; prévoir la sortie ; exécuter ; comparer avec le calcul manuel.

In [ ]:
# Une jointure correcte avant de générer le vrai CSV Kaggle.
sample_demo=pd.DataFrame({'id':[18,12,27]})
pred_demo=pd.DataFrame({'id':[12,18,27],'is_fraud':[.9,.1,.3]})
aligned=sample_demo.merge(pred_demo,on='id',how='left',validate='one_to_one',sort=False)
assert aligned.id.tolist()==[18,12,27]
assert np.allclose(aligned.is_fraud,[.1,.9,.3])
display(aligned)

## Geler et produire le vrai format de soumission

**Point d’arrêt avant le calcul 1.**

1. L’enseignant annonce la question traitée et montre les entrées utilisées.
2. Vous dites ce qui est appris sur le train et ce qui est seulement mesuré.
3. Vous exécutez sans changer le code ; vous repérez une sortie et son unité.
4. Vous expliquez ce que cette sortie permet de conclure et ce qu’elle ne prouve pas.

**Ne pas enchaîner si vous ne pouvez pas nommer la sortie.** Le professeur fait un exemple analogue ; les plus avancés ont une extension facultative.

In [ ]:
from mllab.competition import make_submission
# Choix démonstratif : à remplacer par la configuration décidée avant soumission.
release=make_submission(KEY,model='linear',limit=LIMIT)
submission=pd.read_csv(release/'submission.csv')
assert list(submission)==list(sample) and submission.id.equals(sample.id)
assert submission.is_fraud.between(0,1).all()
print('Soumission locale',release);display(submission.head())
print('Aucun envoi à Kaggle. L’équipe vérifie règles, délai et budget de soumissions avant de téléverser.')

## Contractualiser l’inférence et la livraison

**Point d’arrêt avant le calcul 2.**

1. L’enseignant annonce la question traitée et montre les entrées utilisées.
2. Vous dites ce qui est appris sur le train et ce qui est seulement mesuré.
3. Vous exécutez sans changer le code ; vous repérez une sortie et son unité.
4. Vous expliquez ce que cette sortie permet de conclure et ce qu’elle ne prouve pas.

**Ne pas enchaîner si vous ne pouvez pas nommer la sortie.** Le professeur fait un exemple analogue ; les plus avancés ont une extension facultative.

In [ ]:
from mllab.competition import make_model,xy,subsets
tr,va,_,_,_=subsets(KEY,limit=LIMIT);model=make_model(KEY,tr,'linear').fit(xy(tr,KEY),tr.is_fraud)
one=xy(va,KEY).iloc[:1]
assert model.predict_proba(one).shape==(1,2)
contract={'input_columns':one.columns.tolist(),'dtypes':{k:str(v) for k,v in one.dtypes.items()},'classes':model.classes_.tolist(),'target':SPEC['target'],'metric':SPEC['metric'],'data_hashes':protocol_record['data_sha256']}
display(contract)
print('Refuser schéma invalide ; conserver prétraitement et ordre des colonnes. Ne pas charger un pickle inconnu.')

## Extensions facultatives — après validation du socle

Ces pistes servent aux étudiants déjà à l’aise. Elles ne sont pas à réaliser toutes ni nécessaires pour accéder à la séance suivante. Le travail obligatoire est celui indiqué dans le parcours guidé et la fiche de séance.

1. Faire reproduire le candidat par un autre membre sans assistance des auteurs.
2. Rédiger la fiche modèle et une fonction d’inférence qui refuse un schéma incorrect ; exposer une API locale si le groupe est prêt.
3. Présenter bilan du concours de novembre et plan de transfert ; ne pas promettre que le concours reste ouvert le 17 décembre.

Pour chaque essai : hypothèse avant calcul, code/configuration, empreinte du split, métrique officielle, durée, budget, conclusion, prochaine expérience. Chaque membre doit pouvoir expliquer son apport.

**Point d’arrêt avant le calcul 3.**

1. L’enseignant annonce la question traitée et montre les entrées utilisées.
2. Vous dites ce qui est appris sur le train et ce qui est seulement mesuré.
3. Vous exécutez sans changer le code ; vous repérez une sortie et son unité.
4. Vous expliquez ce que cette sortie permet de conclure et ce qu’elle ne prouve pas.

**Ne pas enchaîner si vous ne pouvez pas nommer la sortie.** Le professeur fait un exemple analogue ; les plus avancés ont une extension facultative.

In [ ]:
HYPOTHESE = 'À écrire avant le prochain calcul'
CONCLUSION = 'À compléter avec une preuve et une limite'
# Ajouter ici une expérience limitée, puis expliquer la décision de la conserver ou non.

## Questions de sortie

- Pourquoi un checkpoint sans code de prétraitement est-il incomplet ?
- Que feras-tu réellement si les données changent ?

**Remise :** un run traçable, une conclusion, un artefact ou une soumission valide et une prochaine hypothèse. Aucun score public ni prix garanti.